### Agent from Scratch

In [1]:
from langchain_aws import ChatBedrock
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

Tools

In [2]:
@tool
def get_product_price(product: str) -> float:
    """Agentic Tool: Get Product Price

    Args:
        product (str): name of the item to search for

    Returns:
        float: return price of a given item
    """
    print(f"    >> Executing get_product_price(product='{product}')")

    prices = {
        "laptop": 1299.99,
        "headphones": 149.95,
        "keyboard": 89.95
    }

    return prices.get(product, 0)

In [3]:
@tool
def apply_discount(price: float, discount_tier: str) -> float:
    """Tool: Apply Discount

    Args:
        price (float): item price
        discount_tier (str): name of tier

    Returns:
        float: return discount rate
    """
    print(f"    >> Executing apply_discount(price: '{price}', discount_tier: '{discount_tier}')")

    discount_percentages = {"bronze": 5, "silver": 12, "gold": 23}
    discount = discount_percentages.get(discount_tier, 0)

    return round(price * (1 - discount / 100), 2)

Agent

In [4]:
def run_agent(question: str):

    # tool and dict of tools
    tools = [get_product_price, apply_discount]
    tools_dict = {t.name: t for t in tools}

    # creating llm and binding the tools
    llm = init_chat_model(f"bedrock:us.anthropic.claude-sonnet-4-20250514-v1:0", temperature=0)
    llm_with_tools = llm.bind_tools(tools)

    # creating question prompt
    print(f"Qustion: {question}")
    print("=" * 60)

    # system messages
    messages = [
        SystemMessage(
            content=(
                "You are a helpful shopping assistant"
                "You have access to a product catalog tool"
                "and a discount tool.\n\n"
                "STRICT RULES - you must follow these exactly:\n"
                "1. Never assume a price"
                "You MUST call get_product_price first to get the real price.\n"
                "2. Only call apply_discount AFTER you have received "
                "a price from get_product_price. Pass the exact price "
                "returned by get_product_price — do NOT pass a made-up number.\n"
                "3. NEVER calculate discounts yourself using math. "
                "Always use the apply_discount tool.\n"
                "4. If the user does not specify a discount tier, "
                "ask them which tier to use — do NOT assume one."
            )
        ),
        HumanMessage(content=question)
    ]

    MAX_ITERS = 10
    for iteration in range(1, MAX_ITERS + 1):
        print(f"--- Iteration {iteration}/{MAX_ITERS} ---")

        ai_messsage = llm_with_tools.invoke(messages)

        tool_calls = ai_messsage.tool_calls

        if not tool_calls:
            print(f"\nFinal Answer: {ai_messsage.content}")
            return ai_messsage.content

        tool_call = tool_calls[0]
        tool_name = tool_call.get("name")
        tool_args = tool_call.get("args", {})
        tool_call_id = tool_call.get("id")

        print(f"    [Tool Selected] {tool_name} with args: {tool_args}")

        tool_to_use = tools_dict.get(tool_name)

        if tool_to_use is None:
            raise ValueError(f"Tool '{tool_name}' not found")

        observation = tool_to_use.invoke(tool_args)
        print(f"    [Tool Result] {observation}")

        messages.append(ai_messsage)
        messages.append(
            ToolMessage(content=str(observation), tool_call_id=tool_call_id)
        )

Run

In [5]:
result = run_agent("What is the price of a laptop after applying a gold discount?")
print(result)

Qustion: What is the price of a laptop after applying a gold discount?
--- Iteration 1/10 ---
    [Tool Selected] get_product_price with args: {'product': 'laptop'}
    >> Executing get_product_price(product='laptop')
    [Tool Result] 1299.99
--- Iteration 2/10 ---
    [Tool Selected] apply_discount with args: {'price': 1299.99, 'discount_tier': 'gold'}
    >> Executing apply_discount(price: '1299.99', discount_tier: 'gold')
    [Tool Result] 1000.99
--- Iteration 3/10 ---

Final Answer: The laptop originally costs $1,299.99. After applying the gold discount, the final price is $1,000.99.
The laptop originally costs $1,299.99. After applying the gold discount, the final price is $1,000.99.
